# OLS regression in Python

We'll start with **ordinary least squares (OLS) regression** -- the workhorse of econometrics and finance. We'll use the `statsmodels` library to run regressions and interpret the output, just like you might using Excel's Data Analysis Toolkit.

We'll also cover **indicator and categorical variables**, which are essential for feature engineering in any regression or machine learning model.

This section will use our **Zillow pricing error** data as the main example.

## Supervised learning and the machine learning process. Hull, Chapter 1.

Chapter 1 of the Hull book starts our discussion of machine learning. Traditional econometrics is about **explaining**. Machine learning is about **predicting**. Roughly speaking.

We saw **unsupervised learning** when looking at clustering. Regression gets us into **supervised learning**. From Hull, Chapter 1:

> Supervised learning is concerned with using data to make **predictions**. In the next section, we will show how a simple regression model can be used to predict salaries. This is an example of supervised learning. In Chapter 3, we will consider how a similar model can be used to  predict house prices. We can distinguish between supervised learning models used to predict a variable that can take a continuum of values (such as an individual’s salary or the price of a house) and supervised learning models that are used for classification.

> The data for supervised learning contains what are referred to as  features and labels. The labels are the values of the target (e.g., the value  of a house or whether a particular loan was repaid). The features are  the variables from which the predictions about the target are made.

Prediction means that we're going to approach things a bit differently. In particular, we are going to think carefully about **in-sample** vs. **out-of-sample** prediction. From Hull, Chapter 1:

> When a data set is used for forecasting or determining a decision strategy, there is a danger that the machine learning model will work well for the data set but will not generalize well to other data. As statisticians have realized for a long time, it is important to test a model out-of-sample. By this we mean that the model should be tested on data that is different from the data used to determine the parameters of the model. 

Our process is going to have us **fit** a model using **training** data. We might use some **validation** data to **fine-tune** the model. Finally, we **evaluate** the model using our **testing** data.


### Supervised machine learning - the basic workflow

A lot of the data that we're using in these examples is already clean. There are no missing values. The columns are what we want. The features and target have been standardized. 

In reality, things don't work that way. You need to **carefully look at your data**. The regressions, the models, the machine learning is "easy", in a sense. The tools are built-in. You don't exactly have to do a lot of linear algebra or optimization math by hand. 

The real work is in the details. It is your judgement and domain expertise that tells you what features you should be looking for. You need to know how your data were collected. You need the background knowledge to understand your task.

You'll spend a lot of time cleaning and organizing your data in our labs and exams. In general, your workflow is going to be something like this:

- **Import your data**. Is it a CSV or Excel file? Is it what you want? Do you need to remove any extra columns? Skip rows? 
  
- **Keep the columns that your want.** What's your target, the thing you're predicting? What features do you want to include?
  
- **Feature engineering time!** Any missing values? Anything to clean? Check all of your columns. Are some text that you want numeric? Everything we're doing here, for now, is with numeric values. Do you need to create new variables from existing ones? Combine variables into a new feature? Do you want to create dummy variables for categorical features? **You'll spend most of your time doing this.**
  
- **Split your data.** Once your data is clean, you can split it into **training** and **testing** samples. Hull will also sometimes split things into an intermediate **validation** sample. Most of your data will be in the training sample. You'll hold out maybe 20% of your data for testing.
  
- **Standardize your data.** Turn the actual numerical values for your targets and features into z-scores. A general recommendation is to standardize your training data and then use the means and standard deviations from your training data to standardize your testing data. More on this below. 
  
Hull often provides data that is already cleaned, split, and standardized. But, we need to know how to do this. He mentions in the text that he standardizes the testing data using the means and standard deviations from the training data. But, we don't see this when using data that is already cleaned up.
  
- **Train your model.** Use your training data to fit your model. Use a validation data set or something like cross-fold validation to look for the optimal hyperparameter(s), if needed.
  
- **Predict your target.** Take the fitted model that uses the optimal hyperparameter(s) and use it to fit your testing data. Get predicted values and compare to the actual y targets in your testing sample.
  
- **Evaluate your model.** How good a job did it do? See Hull for more on the numbers you might use to evaluate a model. I like to make a scatter plot of actual vs. predicted values. 


## What is regression? Hull, Chapter 3.

We are going to run a bunch of regressions. You've seen the basics in other classes. However, we should make sure that we really understand what we're doing. At its core, linear regression attempts to find the best-fitting line (or hyperplane in multiple dimensions) through a set of data points, minimizing the discrepancy between the observed values and the values predicted by the model.


```{note}
Chapters 3.1 and 3.2 of Hull cover regression basics and some of the algebra, with an example for housing prices. If you are doing anything even remotely ``quant`` or data science or analytics for a job, you should know regression really, really well. Be able to derive these equations.
```

This is an excellent overview: <https://aeturrell.github.io/coding-for-economists/econmt-regression.html>. Some important concepts covered:

- Notation and assumptions behind regression.
- Using the `pyfixest` package. We'll use a different package below, but this one is nice.
- Standard errors. They determine the **significance level** of your coefficients and often **need to be fixed**. 
- Fixed effects and categorical variables.
- Transforming your variables with things like logs. Our textbook talks about this when discussing **defining our features**. 
- Causality with things like instrumental variables. Beyond our scope, but businesses are interested in these things. They want to run **experiments** to get at **what happens to Y if we do X?** This is a causal question, but pure statistics. 

I also like this section on **regression diagnostics**: <https://aeturrell.github.io/coding-for-economists/econmt-diagnostics.html>. Does your model do a good job explaining the relationship that you're interested in?
  

Back to the basics. The population linear regression model is written as:


$$y_i = \beta_0 + \beta_1 x_{i1} + \beta_2 x_{i2} + \cdots + \beta_k x_{ik} + u_i$$


or, in matrix form:


$$\mathbf{y} = \mathbf{X} \boldsymbol{\beta} + \mathbf{u}$$


where:

$\mathbf{y}$  is an  $n \times 1$ vector of the dependent variable,
$\mathbf{X}$  is an  $n \times (k+1)$ matrix of regressors (including a column of ones for the intercept),
$\boldsymbol{\beta}$  is a  $(k+1) \times 1$ vector of coefficients,
$\mathbf{u}$  is an  $n \times 1$ vector of error terms.

The goal of ordinary least squares (OLS) is to estimate  $\boldsymbol{\beta}$  such that the sum of squared residuals  $\mathbf{u}{\prime}\mathbf{u}$ is **minimized**. In short, regression is an optimization problem (like you would in calculus) expressed with some linear algebra (so, vectors and matrices).

The OLS estimator is derived as:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^{\prime}\mathbf{X})^{-1} \mathbf{X}^{\prime}\mathbf{y}$$


This estimator gives the “best” linear approximation of the relationship between the variables under certain conditions. The $\prime$ means the **transpose** of the matrix. The ${-1}$ is a matrix **inverse**. You probably saw all of this in high school math. 

### Assumptions and the Gauss-Markov Theorem

The validity of OLS relies on several classical assumptions (often called the Gauss-Markov assumptions):
1. Linearity: The model is linear in parameters.
2. Random Sampling: The data is an i.i.d. sample from the population.
3. No Perfect Multicollinearity: No regressor is a perfect linear combination of others.
4. Zero Conditional Mean: $\mathbb{E}[u_i | \mathbf{X}] = 0$ , implying regressors are exogenous.
5. Homoskedasticity:  $\mathbb{E}[u_i^2 | \mathbf{X}] = \sigma^2$, i.e., constant variance of the error term.

Under these assumptions, the Gauss-Markov Theorem states that the OLS estimator  $\hat{\boldsymbol{\beta}}$ is the **Best Linear Unbiased Estimator (BLUE)**. This means that among all linear and unbiased estimators, OLS has the smallest variance. It's the best!

These assumptions are actually pretty general. OLS works in all kinds of settings, with all kinds of data. It's actually kind of amazing that something this simple is used absolutely everywhere.

```{figure} ../images/12-ols1.png
---
name: 12-ols1.png
align: center
class: with-border
---
You want to understand when some tools are appropriate. But, don't over-complicate things. 
```

But, you still want to understand the assumptions. OLS isn't magic. It's just some math. OLS **doesn't tell you if a relationship is causal**, that one of the X variables causes the Y variable. For all OLS cares, you could switch the Y and X variables around and the math would still work. It's really just telling you **conditional correlations**.

For causality, you need **experimental design**. That's beyond the scope of what we're doing. Take econometrics. 

### Interpreting coefficients and regression output

When you run a regression, especially in Python with `statsmodels`, you’ll get a table of output. Here’s how to make sense of the key parts:
- Coefficients: These are your $\hat{\beta}_j$  values. They measure the estimated effect of each feature on the outcome.
- Standard Errors: These measure the uncertainty around each coefficient estimate. Smaller values mean more precise estimates.
- t-Statistic: Calculated as $\hat{\beta}_j / \text{SE}(\hat{\beta}_j)$. A large absolute value (e.g. greater than 1.96) suggests the coefficient is likely nonzero.
- p-Value: Tells you whether the coefficient is statistically significant. Typically, a p-value < 0.05 is considered evidence against the null (that the coefficient is 0).
- R-squared: The proportion of variance in the dependent variable explained by the model. Closer to 1 means better fit — but beware of overfitting.
- Adjusted R-squared: Like R² but penalizes for too many variables. Useful for comparing models.
- F-statistic: Tests whether all the regression coefficients are jointly equal to zero.

Intuitively, each coefficient $\beta_j$ represents the partial correlation of the corresponding explanatory variable $x_j$ on the dependent variable $y$ , holding all other variables constant. In algebraic terms, in the linear model:

$\frac{\partial y_i}{\partial x_{ij}} = \beta_j$

So if  $\beta_2 = 5$, then a one-unit increase in $x_2$ is associated with a 5-unit increase in the expected value of $y$ , assuming other variables are held constant.

```{figure} ../images/12-ols2.png
---
name: 12-ols2.png
align: center
class: with-border
---
Regression is great.
```

In practice, it’s important to consider:
- The sign: Is the relationship positive or negative?
- The magnitude: Is the change meaningful in the context of the units?
- The statistical significance: Is the estimated effect distinguishable from zero?

Statistical significance is usually assessed using the t-statistic and p-value, which tell us how confident we are that the effect is real (i.e., unlikely due to chance under the null hypothesis that $\beta_j = 0$).

Once you’ve estimated your regression, it’s good practice to check how well the model is doing. One way to do this is by examining the residuals — the differences between the actual and predicted values:

$\hat{u}_i = y_i - \hat{y}_i$

I'll do something like this below, when I compare the actual vs. predicted values.

Residuals should:
- Be randomly scattered (no pattern!) when plotted against fitted values or individual predictors.
- Have roughly constant variance (homoskedasticity).
- Be centered around zero.

If these aren’t true, your model may be missing something — a nonlinear pattern, a missing variable, or a transformation (like taking logs) might help.

Chapter 10.1 discusses how to interpret our results. Traditionally, machine learning hasn't been as interested in interpreting the model - we just want predicted values! However, as machine learning, statistics, econometrics, and specific domain knowledge (i.e. finance) mix, we are becoming more interested in exactly how to interpret our results.

As the text notes, you can't tell a loan applicant that they got rejected because "the algo said so"! You need to know why. This also gets into biases in our data. Is your model doing something illegal and/or unethical?

From the Hull text:

> The weight, $b_j$, can be interpreted as the sensitivity of the prediction to the value of the feature $j$. If the value of feature $j$ increases by an amount $u$ with all other features remaining the same, the value of the target increases by $b_{j,u}$. In the case of categorical features that are 1 or 0, the  weight gives the impact on a prediction of the target of changing the  category of the feature when all other features are kept the same. 

> The bias, $a$, is a little more difficult to interpret. It is the value of the target if all the feature values are zero. However, feature values of zero make no sense in many situations.

The text suggests setting all of the X features at the average value and then adding the intercept, $a$. This $a^*$ value is the predicted y if all features are at their mean. 

Note that to do this, you should use the model results with the unscaled data. From the text:

> To improve interpretability, the Z-score scaling used for Table 3.7 has been reversed in Table 10.1. (It will be recalled that scaling was necessary for Lasso.) This means that the feature weights in Table 3.7 are divided by the standard deviation of the feature and multiplied by the standard deviation of the house price to get the feature weights in Table 10.1.

As mentioned above, going back and forth between scaled and unscaled features is easier if you just let `sklearn` do it for you.

This is also a [nice summary](https://scikit-learn.org/stable/auto_examples/inspection/plot_linear_model_coefficient_interpretation.html#sphx-glr-auto-examples-inspection-plot-linear-model-coefficient-interpretation-py) of how to interpret regression results.


### Regression and common interview questions

If you are interviewing for any kind of data science or analytics job, you should be able to answer some basic questions. You might know the ones for careers like banking (e.g. Walk me through  a DCF). There are, of course, the important behavioral questions. Here are some common questions for more data-oriented jobs.


- What is linear regression? How does it work?
- What is the difference between correlation and causation?
- What is multicollinearity? How do you detect it?
- What is heteroskedasticity? How do you detect it?
- What is the difference between a linear and a logistic regression?
- What is the difference between LASSO and Ridge regression?
- What is the difference between supervised and unsupervised learning?
- How do you interpret the coefficients in a regression model?
- What is the purpose of regularization in regression models?
- What is the purpose of cross-validation in regression models?
- What is the purpose of feature selection in regression models?

We touch on all of these topics below.


### statsmodel vs. sklearn

I run linear regressions (OLS) two ways below. I first use `statsmodels`. This method and output looks like "traditional" regression analysis. You specify the y-variable, the x-variables you want to include, and get a nice table with output. Very much like what you'd get using Excel and the Data Analysis Toolkit. 

The Hull textbook uses `sklearn`. This library is focused on **machine learning**. The set-up is going to look a lot different. This library will help you process your data, define features, and split it up into training and test data sets. Crucially, you'll want to end up with **X** and **y** data frames that contain **features** and **target** values that you are trying to predict, respectively.

Two libraries, two ways to do regression. The Hull textbook uses `sklearn` for the **ridge, LASSO, and Elastic Net** models.

You can read more about [the statsmodels library on their help page](https://www.statsmodels.org/dev/regression.html). Here's [linear regression from the sklearn library](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). 

## Example using Zillow pricing errors

Let's start with a typical regression and output. So, no prediction. Let's run a regression like we might using the Data Analysis Toolkit in Excel. We'll look at the output and interpret the coefficients and statistical significance. 

I'll be using our Zillow pricing error data in this example. The `statsmodels` library will let us run the regressions and give us nicely formatted output.

In [1]:
import numpy as np
import scipy.stats as st
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import r2_score as r2
from sklearn.linear_model import LinearRegression

# Importing Ridge
from sklearn.linear_model import Ridge

# Import Lasso
from sklearn.linear_model import Lasso

# Import Elastic Net
from sklearn.linear_model import ElasticNet

# Import train_test_split
from sklearn.model_selection import train_test_split

# Import StandardScaler
from sklearn.preprocessing import StandardScaler

# Import GridSearchCV
from sklearn.model_selection import GridSearchCV

#Need for AIC/BIC cross validation example
from sklearn.linear_model import LassoLarsIC
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LassoCV
from sklearn.model_selection import RepeatedKFold


# Include this to have plots show up in your Jupyter notebook.
%matplotlib inline 

pd.options.display.max_columns = None

In [2]:
housing = pd.read_csv('https://raw.githubusercontent.com/aaiken1/fin-data-analysis-python/main/data/properties_2016_sample10_1.csv')
pricing = pd.read_csv('https://raw.githubusercontent.com/aaiken1/fin-data-analysis-python/main/data/train_2016_v2.csv')

zillow_data = pd.merge(housing, pricing, how='inner', on='parcelid')
zillow_data['transactiondate'] = pd.to_datetime(zillow_data['transactiondate'], format='%Y-%m-%d')


/var/folders/kx/y8vj3n6n5kq_d74vj24jsnh40000gn/T/ipykernel_46842/4278326240.py:1: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  housing = pd.read_csv('https://raw.githubusercontent.com/aaiken1/fin-data-analysis-python/main/data/properties_2016_sample10_1.csv')


In [3]:
zillow_data.describe()

,parcelid,airconditioningtypeid,architecturalstyletypeid,basementsqft,bathroomcnt,bedroomcnt,buildingclasstypeid,buildingqualitytypeid,calculatedbathnbr,decktypeid,finishedfloor1squarefeet,calculatedfinishedsquarefeet,finishedsquarefeet12,finishedsquarefeet13,finishedsquarefeet15,finishedsquarefeet50,finishedsquarefeet6,fips,fireplacecnt,fullbathcnt,garagecarcnt,garagetotalsqft,heatingorsystemtypeid,latitude,longitude,lotsizesquarefeet,poolcnt,poolsizesum,pooltypeid7,propertylandusetypeid,rawcensustractandblock,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,storytypeid,threequarterbathnbr,typeconstructiontypeid,unitcnt,yardbuildingsqft17,yardbuildingsqft26,yearbuilt,numberofstories,structuretaxvaluedollarcnt,taxvaluedollarcnt,assessmentyear,landtaxvaluedollarcnt,taxamount,taxdelinquencyyear,censustractandblock,logerror
count,9.071000e+03,2871.000000,0.0,5.00000,9071.000000,9071.000000,3.0,5694.000000,8948.000000,64.0,695.000000,9001.000000,8612.000000,3.000000,337.000000,695.000000,49.000000,9071.000000,993.000000,8948.000000,3076.000000,3076.000000,5574.000000,9.071000e+03,9.071000e+03,8.020000e+03,1810.0,99.000000,1685.0,9071.000000,9.071000e+03,8912.000000,9071.000000,3601.000000,9070.000000,9071.000000,5.0,1208.000000,0.0,5794.000000,280.000000,7.000000,8991.000000,2138.000000,9.022000e+03,9.071000e+03,9071.0,9.071000e+03,9071.000000,168.000000,9.009000e+03,9071.000000
mean,1.298764e+07,1.838036,NaN,516.00000,2.266233,3.013670,4.0,5.572708,2.296826,66.0,1348.981295,1767.239307,1740.108918,1408.000000,2393.350148,1368.942446,2251.428571,6049.128982,1.197382,2.228990,1.800715,342.415475,3.909760,3.400230e+07,-1.181977e+08,3.150909e+04,1.0,520.424242,1.0,261.835520,6.049436e+07,33944.006845,2511.879727,193520.398223,96547.689195,1.531364,7.0,1.004967,NaN,1.104764,290.335714,496.714286,1968.380047,1.428438,1.768673e+05,4.523049e+05,2015.0,2.763930e+05,5906.696988,13.327381,6.049368e+13,0.010703
std,1.757451e+06,3.001723,NaN,233.49197,0.989863,1.118468,0.0,1.908379,0.960557,0.0,664.508053,918.999586,880.213401,55.425626,1434.457485,709.622839,1352.034747,20.794593,0.480794,0.951007,0.598328,263.642761,3.678727,2.654493e+05,3.631575e+05,1.824345e+05,0.0,146.537109,0.0,5.781663,2.063550e+05,47178.373342,810.417898,169701.596819,412.732130,2.856603,0.0,0.070330,NaN,0.459551,172.987812,506.445033,23.469997,0.536698,1.909207e+05,5.229433e+05,0.0,3.901131e+05,6388.966672,1.796527,2.053649e+11,0.158364
min,1.071186e+07,1.000000,NaN,162.00000,0.000000,0.000000,4.0,1.000000,1.000000,66.0,49.000000,214.000000,214.000000,1344.000000,716.000000,49.000000,438.000000,6037.000000,1.000000,1.000000,0.000000,0.000000,1.000000,3.334420e+07,-1.194143e+08,4.350000e+02,1.0,207.000000,1.0,31.000000,6.037101e+07,3491.000000,1286.000000,6952.000000,95982.000000,0.000000,7.0,1.000000,NaN,1.000000,41.000000,37.000000,1885.000000,1.000000,1.516000e+03,7.837000e+03,2015.0,2.178000e+03,96.740000,7.000000,6.037101e+13,-2.365000
25%,1.157119e+07,1.000000,NaN,485.00000,2.000000,2.000000,4.0,4.000000,2.000000,66.0,938.000000,1187.000000,1173.000000,1392.000000,1668.000000,938.000000,1009.000000,6037.000000,1.000000,2.000000,2.000000,0.000000,2.000000,3.380545e+07,-1.184080e+08,5.746500e+03,1.0,435.000000,1.0,261.000000,6.037400e+07,12447.000000,1286.000000,46736.000000,96193.000000,0.000000,7.0,1.000000,NaN,1.000000,175.750000,110.500000,1953.000000,1.000000,8.028525e+04,1.926595e+05,2015.0,8.060700e+04,2828.645000,13.000000,6.037400e+13,-0.025300
50%,1.259048e+07,1.000000,NaN,515.00000,2.000000,3.000000,4.0,7.000000,2.000000,66.0,1249.000000,1539.000000,1513.000000,1440.000000,2157.000000,1257.000000,1835.000000,6037.000000,1.000000,2.000000,2.000000,430.000000,2.000000,3.401408e+07,-1.181670e+08,7.200000e+03,1.0,504.000000,1.0,261.000000,6.037621e+07,25218.000000,3101.000000,118887.000000,96401.000000,0.000000,7.0,1.000000,NaN,1.000000,248.500000,268.000000,1969.000000,1.000000,1.315530e+05,3.416920e+05,2015.0,1.910000e+05,4521.580000,1

I'll print a list of the columns, just to see what our variables are. There's a lot in this data set.

In [4]:
zillow_data.columns

Index(['parcelid', 'airconditioningtypeid', 'architecturalstyletypeid',
       'basementsqft', 'bathroomcnt', 'bedroomcnt', 'buildingclasstypeid',
       'buildingqualitytypeid', 'calculatedbathnbr', 'decktypeid',
       'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet',
       'finishedsquarefeet12', 'finishedsquarefeet13', 'finishedsquarefeet15',
       'finishedsquarefeet50', 'finishedsquarefeet6', 'fips', 'fireplacecnt',
       'fullbathcnt', 'garagecarcnt', 'garagetotalsqft', 'hashottuborspa',
       'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet',
       'poolcnt', 'poolsizesum', 'pooltypeid10', 'pooltypeid2', 'pooltypeid7',
       'propertycountylandusecode', 'propertylandusetypeid',
       'propertyzoningdesc', 'rawcensustractandblock', 'regionidcity',
       'regionidcounty', 'regionidneighborhood', 'regionidzip', 'roomcnt',
       'storytypeid', 'threequarterbathnbr', 'typeconstructiontypeid',
       'unitcnt', 'yardbuildingsqft17', 'yardbuildin

Let's run a really simple regression. Can we explain pricing errors using the size of the house? I'll take the natural log of `calculatedfinishedsquarefeet` and use that as my independent (**X**) variable. My dependent (**Y**) variable will be `logerror`. I'm taking the natural log of the square footage, in order to have what's called a "log-log" model.

In [5]:
zillow_data['ln_calculatedfinishedsquarefeet'] = np.log(zillow_data['calculatedfinishedsquarefeet'])

results = smf.ols("logerror ~ ln_calculatedfinishedsquarefeet", data=zillow_data).fit()


In [6]:
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               logerror   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     13.30
Date:                Sat, 20 Apr 2024   Prob (F-statistic):           0.000267
Time:                        14:28:55   Log-Likelihood:                 3831.8
No. Observations:                9001   AIC:                            -7660.
Df Residuals:                    8999   BIC:                            -7645.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

One y variable and one X variable. That's the full summary of the regression. This is a "log-log" model, so we can say that a 1% change in square footage leads to a 1.39% increase in pricing error. Not price! Pricing error. The coefficient is positive and statistically significant at conventional levels (e.g. 1%). 

```{note}
You'll see terms like log-log, log-linear, and linear-log. These are all different ways of transforming your variables. The log-log model is the most common. It means that both the dependent and independent variables are in logs. A log-linear model means that the dependent variable is in logs, but the independent variable is not. A linear-log model means that the dependent variable is not in logs, but the independent variable is. For example, if our example was log-linear, we would get the percentage change in pricing error for a one-unit change in square footage.
```

And, the Y and X variables are not arbitrary. We are trying to explain the pricing error (Y) using the size of the house (X). You could swap them - the math doesn't care. But, we do! However, just because you do some math doesn't mean that you have a good model of causality. That requires **design**.  
We can pull out just a piece of this full result if we like. 

In [7]:
results.summary().tables[1]

,coef,std err,t,P>|t|,[0.025,0.975]
Intercept,-0.0911,0.028,-3.244,0.001,-0.146,-0.036
ln_calculatedfinishedsquarefeet,0.0139,0.004,3.647,0.000,0.006,0.021


We can, of course, include multiple **X** variables in a regression. I'll add bathroom and bedroom counts to the regression model.

Pay attention to the **syntax** here. I am giving `smf.ols` the name of my data frame. I can then **write the formula** for my regression using the names of my columns (variables or features).

In [8]:
results = smf.ols("logerror ~ ln_calculatedfinishedsquarefeet + bathroomcnt + bedroomcnt", data=zillow_data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               logerror   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     6.718
Date:                Sat, 20 Apr 2024   Prob (F-statistic):           0.000159
Time:                        14:28:55   Log-Likelihood:                 3835.2
No. Observations:                9001   AIC:                            -7662.
Df Residuals:                    8997   BIC:                            -7634.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

Hey, all of my significance went away! Welcome to the world of [multicollinearity](https://en.wikipedia.org/wiki/Multicollinearity). All of these variables are very correlated, so the coefficient estimates become difficult to interpret.

We're going to use **machine learning** below to help with this issue.

Watch what happens when I just run the model with the bedroom count. The $t$-statistic is quite large again.

In [9]:
results = smf.ols("logerror ~ bedroomcnt", data=zillow_data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               logerror   R-squared:                       0.002
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     21.69
Date:                Sat, 20 Apr 2024   Prob (F-statistic):           3.24e-06
Time:                        14:28:55   Log-Likelihood:                 3856.7
No. Observations:                9071   AIC:                            -7709.
Df Residuals:                    9069   BIC:                            -7695.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0101      0.005     -2.125      0.0

### Indicators and categorical variables

The variables used above are measured numerically. Some are **continuous**, like square footage, while others are **counts**, like the number of bedrooms. Sometimes, though, we want to include an **indicator** for something. For example, does this house have a pool or not?

These kinds of variables are called **categorical** and are a very common way to structure your features. You can read more here: <https://aeturrell.github.io/coding-for-economists/data-categorical.html>

There is a variable in the data called `poolcnt`. It seems to be either missing (NaN) or set equal to 1. I believe that a value of 1 means that the house has a pool and that `NaN` means that it does not. This is bit of a tricky assumption, because `NaN` could mean no pool or that we don't know either way. But, I'll make that assumption for illustrative purposes.

In [10]:
zillow_data['poolcnt'].describe()

count    1810.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: poolcnt, dtype: float64

I am going to create a new variable, `pool_d`, that is set equal to 1 if `poolcnt >= 1` and 0 otherwise. This type of 1/0 categorical variable is sometimes called an **indicator**, or **dummy** variable. 

This is an example of making the indicator variable **by hand**. I'll use `pd.get_dummies` below in a second.

In [11]:
zillow_data['pool_d'] = np.where(zillow_data.poolcnt.isna(), 0, zillow_data.poolcnt >= 1)
zillow_data['pool_d'].describe()

count    9071.000000
mean        0.199537
std         0.399674
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: pool_d, dtype: float64

I can then use this 1/0 variable in my regression.

In [12]:
results = smf.ols("logerror ~ ln_calculatedfinishedsquarefeet + pool_d", data=zillow_data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               logerror   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     6.684
Date:                Sat, 20 Apr 2024   Prob (F-statistic):            0.00126
Time:                        14:28:55   Log-Likelihood:                 3831.8
No. Observations:                9001   AIC:                            -7658.
Df Residuals:                    8998   BIC:                            -7636.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

Pools don't seem to influence pricing errors. 

We can also create more general **categorical** variables. For example, instead of treating bedrooms like a count, we can create new categories for each number of bedrooms. This type of model is helpful when dealing with states or regions. For example, you could turn a zip code into a categorical variable. This would allow zip codes, or a location, to explain the pricing errors. 

When using `statsmodels` to run your regressions, you can turn something into a categorical variable by using `C()` in the regression formula. 

I'll try the count of bedrooms first.

In [13]:
results = smf.ols("logerror ~ ln_calculatedfinishedsquarefeet + C(bedroomcnt)", data=zillow_data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               logerror   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     3.118
Date:                Sat, 20 Apr 2024   Prob (F-statistic):           0.000196
Time:                        14:28:55   Log-Likelihood:                 3843.8
No. Observations:                9001   AIC:                            -7662.
Df Residuals:                    8988   BIC:                            -7569.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

And here are zip codes as a **categorical variable**. This is saying: Is the house in this zip code or no? If it is, the indicator for that zip code gets a 1, and a 0 otherwise. If we didn't do this, then the zip code would get treated like a numerical variable in the regression, like square footage, which makes no sense!

In [14]:
results = smf.ols("logerror ~ ln_calculatedfinishedsquarefeet + C(regionidzip)", data=zillow_data).fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               logerror   R-squared:                       0.054
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     1.300
Date:                Sat, 20 Apr 2024   Prob (F-statistic):           0.000104
Time:                        14:28:56   Log-Likelihood:                 4075.3
No. Observations:                9001   AIC:                            -7391.
Df Residuals:                    8621   BIC:                            -4691.
Df Model:                         379                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
Intercept 

**Think about these categorical variables as switches**. We also call them **dummy or indicator variables** when they are constructed in a 1/0 manner. If the variable is 1, then that observation has that characteristic. The predicted value changes by the coefficient if the indicator is 1. The switch is "turned on". 


### Using get_dummies

`pandas` also has a method called `pd.get_dummies`. Here's a description from Claude (my emphasis):

pd.get_dummies() is a function in the Pandas library that is used to transform categorical variables into a format that can be used in machine learning models.

The function takes a Pandas DataFrame or Series as input and creates a new DataFrame where each unique categorical value is represented as a new column. The new columns are binary, with a value of 1 if the observation had that categorical value, and 0 otherwise.

This is a common preprocessing step for many machine learning algorithms that require numeric input features. By converting categorical variables into this dummy or one-hot encoded format, the machine learning model can better understand and incorporate the information from those categorical variables.

For example, if you had a DataFrame with a 'color' column that contained the values 'red', 'green', and 'blue', pd.get_dummies() would create three new columns: 'color_red', 'color_green', and 'color_blue', each with 0s and 1s indicating the color of each observation.

**This transformation allows the machine learning model to treat these categorical variables as distinct features, rather than trying to interpret them as ordinal or numeric data**. Using pd.get_dummies() is a crucial step in preparing many datasets for modeling.

One thing to note - it creates a new dataframe that you need to add back to the original data.

Here's some code, also using *bedroomcnt*.

In [15]:
# Create categorical variable based on bedroomcnt
bedroomcnt_categorical = pd.get_dummies(zillow_data['bedroomcnt'], prefix='bedroomcnt')

# Concatenate the categorical variable with the original DataFrame
zillow_data = pd.concat([zillow_data, bedroomcnt_categorical], axis=1)

I created the new columns and added them to the original Zillow dataframe. Each is an indicator (1/0) variable for whether or not the house has that number of bedrooms. If it does, we flip the switch and the coefficient tells us if a house with that number of bedrooms has larger or smaller pricing errors.

In [16]:
zillow_data.columns

Index(['parcelid', 'airconditioningtypeid', 'architecturalstyletypeid',
       'basementsqft', 'bathroomcnt', 'bedroomcnt', 'buildingclasstypeid',
       'buildingqualitytypeid', 'calculatedbathnbr', 'decktypeid',
       'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet',
       'finishedsquarefeet12', 'finishedsquarefeet13', 'finishedsquarefeet15',
       'finishedsquarefeet50', 'finishedsquarefeet6', 'fips', 'fireplacecnt',
       'fullbathcnt', 'garagecarcnt', 'garagetotalsqft', 'hashottuborspa',
       'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet',
       'poolcnt', 'poolsizesum', 'pooltypeid10', 'pooltypeid2', 'pooltypeid7',
       'propertycountylandusecode', 'propertylandusetypeid',
       'propertyzoningdesc', 'rawcensustractandblock', 'regionidcity',
       'regionidcounty', 'regionidneighborhood', 'regionidzip', 'roomcnt',
       'storytypeid', 'threequarterbathnbr', 'typeconstructiontypeid',
       'unitcnt', 'yardbuildingsqft17', 'yardbuildin

In [17]:
zillow_data

,parcelid,airconditioningtypeid,architecturalstyletypeid,basementsqft,bathroomcnt,bedroomcnt,buildingclasstypeid,buildingqualitytypeid,calculatedbathnbr,decktypeid,finishedfloor1squarefeet,calculatedfinishedsquarefeet,finishedsquarefeet12,finishedsquarefeet13,finishedsquarefeet15,finishedsquarefeet50,finishedsquarefeet6,fips,fireplacecnt,fullbathcnt,garagecarcnt,garagetotalsqft,hashottuborspa,heatingorsystemtypeid,latitude,longitude,lotsizesquarefeet,poolcnt,poolsizesum,pooltypeid10,pooltypeid2,pooltypeid7,propertycountylandusecode,propertylandusetypeid,propertyzoningdesc,rawcensustractandblock,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,storytypeid,threequarterbathnbr,typeconstructiontypeid,unitcnt,yardbuildingsqft17,yardbuildingsqft26,yearbuilt,numberofstories,fireplaceflag,structuretaxvaluedollarcnt,taxvaluedollarcnt,assessmentyear,landtaxvaluedollarcnt,taxamount,taxdelinquencyflag,taxdelinquencyyear,censustractandblock,logerror,transactiondate,ln_calculatedfinishedsquarefeet,pool_d,bedroomcnt_0.0,bedroomcnt_1.0,bedroomcnt_2.0,bedroomcnt_3.0,bedroomcnt_4.0,bedroomcnt_5.0,bedroomcnt_6.0,bedroomcnt_7.0,bedroomcnt_8.0,bedroomcnt_9.0,bedroomcnt_10.0,bedroomcnt_12.0
0,13005045,NaN,NaN,NaN,3.0,2.0,NaN,7.0,3.0,NaN,NaN,1798.0,1798.0,NaN,NaN,NaN,NaN,6037.0,NaN,3.0,NaN,NaN,NaN,2.0,34091843.0,-118047759.0,7302.0,NaN,NaN,NaN,NaN,NaN,0100,261.0,TCR17200*,6.037432e+07,14111.0,3101.0,NaN,96517.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,1936.0,NaN,NaN,119366.0,162212.0,2015.0,42846.0,2246.17,NaN,NaN,6.037432e+13,0.0962,2016-05-18,7.494430,0,0,0,1,0,0,0,0,0,0,0,0,0
1,17279551,NaN,NaN,NaN,3.0,4.0,NaN,NaN,3.0,NaN,1387.0,2302.0,2302.0,NaN,NaN,1387.0,NaN,6111.0,1.0,3.0,2.0,671.0,NaN,NaN,34227297.0,-118857914.0,7258.0,1.0,500.0,NaN,NaN,1.0,1111,261.0,NaN,6.111006e+07,34278.0,2061.0,NaN,96383.0,8.0,NaN,NaN,NaN,NaN,247.0,NaN,1980.0,2.0,NaN,324642.0,541069.0,2015.0,216427.0,5972.72,NaN,NaN,6.111006e+13,0.0020,2016-09-02,7.741534,1,0,0,0,0,1,0,0,0,0,0,0,0
2,12605376,NaN,NaN,NaN,2.0,3.0,NaN,7.0,2.0,NaN,NaN,1236.0,1236.0,NaN,NaN,NaN,NaN,6037.0,NaN,2.0,NaN,NaN,NaN,7.0,33817098.0,-118283644.0,5076.0,NaN,NaN,NaN,NaN,NaN,0100,261.0,CARS*,6.037544e+07,10723.0,3101.0,NaN,96229.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,1957.0,NaN,NaN,167010.0,375907.0,2015.0,208897.0,5160.90,NaN,NaN,6.037544e+13,-0.0566,2016-09-28,7.119636,0,0,0,0,1,0,0,0,0,0,0,0,0
3,11713859,NaN,NaN,NaN,2.0,2.0,NaN,4.0,2.0,NaN,NaN,1413.0,1413.0,NaN,NaN,NaN,NaN,6037.0,NaN,2.0,NaN,NaN,NaN,2.0,34007703.0,-118347262.0,7725.0,NaN,NaN,NaN,NaN,NaN,0100,261.0,LAR1,6.037236e+07,12447.0,3101.0,268097.0,95989.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,1953.0,NaN,NaN,232690.0,588746.0,2015.0,356056.0,7353.80,NaN,NaN,6.037236e+13,0.0227,2016-02-04,7.253470,0,0,0,1,0,0,0,0,0,0,0,0,0
4,17193642,NaN,NaN,NaN,3.5,3.0,NaN,NaN,3.5,NaN,2878.0,2878.0,2878.0,NaN,NaN,2878.0,NaN,6111.0,1.0,3.0,2.0,426.0,NaN,NaN,34177668.0,-118980561.0,10963.0,NaN,NaN,NaN,NaN,NaN,1111,261.0,NaN,6.111006e+07,34278.0,2061.0,46736.0,96351.0,8.0,NaN,1.0,NaN,NaN,312.0,NaN,2003.0,1.0,NaN,392869.0,777041.0,2015.0,384172.0,8668.90,NaN,NaN,6.111006e+13,0.0237,2016-06-28,7.964851,0,0,0,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9066,12653293,NaN,NaN,NaN,3.0,5.0,NaN,7.0,3.0,NaN,NaN,2465.0,2465.0,NaN,NaN,NaN,NaN,6037.0,NaN,3.0,NaN,NaN,NaN,2.0,33726741.0,-118301006.0,4999.0,NaN,NaN,NaN,NaN,NaN,0100,261.0,LAR1,6.037297e+07,12447.0,3101.0,54300.0,96221.0,0.0,NaN,NaN,NaN,1.0,NaN,NaN,1931.0,NaN,NaN,271604.0,559101.0,2015.0,287497.0,6857.67,NaN,NaN,6.037297e+13,-0.0305,2016-03-18,7.809947,0,0,0,0,0,0,1,0,0,0,0,0,0
9067,11907619,1.0,NaN,NaN,3.0,3.0,NaN,4.0,3.0,NaN,NaN,1726.0,1726.0,NaN,NaN,NaN,NaN,6037.0,NaN,3.0,NaN,NaN,NaN,2.0,34072600.0,-118142000.0,187293.0,1.0,NaN,NaN,NaN,1.0,010C,266.0,A

If you are creating a feature data frame that **only includes exactly what you need**, you should drop the original variable once you've created the indicators that you need. In this case, you would drop *bedroomcnt*. 

`pd.get_dummies` has an option `dtype=` for the output. You can choose to get **booleans** (True or False) or **integers** (1/0) for the output. You'll want integers for your regression models. 